In [1]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
load_dotenv()

True

In [2]:
search_tool = DuckDuckGoSearchRun()

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


tools = [search_tool, multiply]


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

llm_with_tools = llm.bind_tools(tools)

In [3]:
class LLMState(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot_node(state: LLMState):

    response = llm_with_tools.invoke(state["messages"])

    return {"messages": [response]}


def human_approval_node(state: LLMState):

    last_message = state["messages"][-1]

    tool_calls = last_message.tool_calls

    approval = interrupt(
        {
            "question": "Approve tool execution?",
            "tool_calls": tool_calls
        }
    )

    return {
        "approved": approval
    }


def should_continue(state: LLMState):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "human_approval"

    return END

def approval_router(state):

    if state["approved"]:
        return "tools"

    return END


tool_node = ToolNode(tools=tools)


In [4]:
builder = StateGraph(LLMState)

builder.add_node("chatbot", chatbot_node)
builder.add_node("human_approval", human_approval_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "chatbot")

builder.add_conditional_edges(
    "chatbot",
    should_continue,
    {
        "human_approval": "human_approval",
        END: END
    }
)

builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "tools": "tools",
        END: END
    }
)

builder.add_edge("tools", "chatbot")

In [5]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

while True:

    user_input = input("\nYou: ")

    result = graph.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        },
        config=config
    )

    if "__interrupt__" in result:

        print("\nTool wants approval!")

        approval = input("Approve? (yes/no): ")

        if approval == "yes":

            result = graph.invoke(
                Command(resume=True),
                config=config
            )

        else:

            result = graph.invoke(
                Command(resume=False),
                config=config
            )

    print("\nAI:", result["messages"][-1].content)


Tool wants approval!

AI: The product of 2 and 3 is 6.

Tool wants approval!

AI: 

AI: Goodbye!


c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_google_genai\chat_models.py:2882: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(



AI: 

AI: 
